# GVH Diagonal Cubic 0.3.2.7.3.7.2.6 — Analytic Schur-Sector Decomposition and Generic Six-Dimensional Inverse

**Auteur :** Charlemagne O Laurince

## Mission

Fermer le dernier verrou de `7.7.2.5` :
\[
\boxed{S_6^{-1}\ \text{générique}}
\]
puis reconstruire \(Q_{\rm total}^{-1}\) par complément de Schur.

On aligne localement
\[
v_i=(0,0,r),\qquad r^2=v_iv^i,
\]
et on décompose le secteur métrique symétrique dans la base
\[
(T,Z,\Delta,K_{12},K_{13},K_{23}),
\]
où
\[
T=K_{11}+K_{22},\quad Z=K_{33},\quad \Delta=K_{11}-K_{22}.
\]

La règle d'audit reste stricte : `RDD2_computed=True` seulement si l'inverse sectorielle est effectivement obtenue analytiquement.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [1]:
import sympy as sp, json, sys
from pathlib import Path
print("GVH 0.3.2.7.3.7.2.6")
print("Python:",sys.version.split()[0])
print("SymPy:",sp.__version__)


GVH 0.3.2.7.3.7.2.6
Python: 3.12.13
SymPy: 1.14.0


## 1. Hessien total aligné et complément de Schur


In [2]:
c1,c2,c3,c4,s,r=sp.symbols("c1 c2 c3 c4 s r", real=True)
K11,K22,K33,K12,K13,K23,S,W1,W2,W3=sp.symbols(
    "K11 K22 K33 K12 K13 K23 S W1 W2 W3", real=True)
vel=[K11,K22,K33,K12,K13,K23,S,W1,W2,W3]

K=sp.Matrix([[K11,K12,K13],[K12,K22,K23],[K13,K23,K33]])
v=sp.Matrix([0,0,r]); W=sp.Matrix([W1,W2,W3])

A=-S
Bv=W-K*v
Cv=-K*v
D=s*K

I1=sp.expand(A**2-Bv.dot(Bv)-Cv.dot(Cv)+sum(D[i,j]**2 for i in range(3) for j in range(3)))
theta=sp.expand(-A+sp.trace(D))
I3=sp.expand(A**2-2*Bv.dot(Cv)+sum(D[i,j]*D[j,i] for i in range(3) for j in range(3)))
alpha=sp.expand(s*A+v.dot(Cv))
beta=sp.expand(s*Bv+D.T*v)
a2=sp.expand(-alpha**2+beta.dot(beta))
Lu=sp.expand(-c1*I1-c2*theta**2-c3*I3+c4*a2)
Qu=sp.hessian(Lu,vel)

LEH=sp.expand(sum(K[i,j]**2 for i in range(3) for j in range(3))-sp.trace(K)**2)
QEH=sp.zeros(10,10)
QEH6=sp.hessian(LEH,vel[:6])
for i in range(6):
    for j in range(6):
        QEH[i,j]=QEH6[i,j]

Q=QEH+Qu
A6=Q[:6,:6]
B64=Q[:6,6:10]
D4=Q[6:10,6:10]
D4inv=D4.inv()
S6=sp.simplify(A6-B64*D4inv*B64.T)

assert Q==Q.T
print("Q_total shape =",Q.shape)
print("S6 shape =",S6.shape)
print("PASS")


Q_total shape = (10, 10)
S6 shape = (6, 6)
PASS


## 2. Décomposition \(SO(2)\) exacte

Avec
\[
K_{11}=\frac{T+\Delta}{2},\qquad
K_{22}=\frac{T-\Delta}{2},
\]
on calcule
\[
S_{\rm sec}=M^TS_6M.
\]

Le résultat doit être
\[
S_0\oplus\lambda_\Delta\oplus\lambda_{12}\oplus\lambda_V\oplus\lambda_V.
\]


In [3]:
M=sp.zeros(6,6)
M[0,0]=sp.Rational(1,2); M[0,2]=sp.Rational(1,2)
M[1,0]=sp.Rational(1,2); M[1,2]=-sp.Rational(1,2)
M[2,1]=1
M[3,3]=M[4,4]=M[5,5]=1

Ssec=sp.simplify(M.T*S6*M)

for i in range(6):
    for j in range(6):
        if not ((i<2 and j<2) or i==j):
            assert sp.simplify(Ssec[i,j])==0

S0=sp.Matrix(Ssec[:2,:2])
lamD=sp.factor(Ssec[2,2])
lam12=sp.factor(Ssec[3,3])
lamV=sp.factor(Ssec[4,4])

assert sp.simplify(Ssec[5,5]-lamV)==0
assert sp.simplify(lam12-4*lamD)==0

print("SO(2) decomposition: PASS")
print("lambda_Delta =",lamD)
print("lambda_12 =",lam12)
print("lambda_V =",lamV)


SO(2) decomposition: PASS
lambda_Delta = -c1*s**2 - c3*s**2 + 1
lambda_12 = -4*(c1*s**2 + c3*s**2 - 1)
lambda_V = 2*(c1**2*r**2 - 2*c1**2*s**2 - 2*c1*c3*s**2 + 2*c1*c4*r**2*s**2 - 2*c1*c4*s**4 + 2*c1 - c3**2*r**2 + 2*c3*c4*r**2*s**2 - 2*c3*c4*s**4 + 2*c4*s**2)/(c1 + c4*s**2)


## 3. Inverse analytique du secteur scalaire \(2\times2\)

Si
\[
S_0=\begin{pmatrix}a&b\\b&d\end{pmatrix},
\qquad
\Delta_0=ad-b^2,
\]
alors
\[
S_0^{-1}
=
\Delta_0^{-1}
\begin{pmatrix}d&-b\\-b&a\end{pmatrix}.
\]


In [4]:
a=sp.factor(S0[0,0])
b=sp.factor(S0[0,1])
d=sp.factor(S0[1,1])
Delta0=sp.factor(a*d-b**2)

S0inv=sp.Matrix([[d,-b],[-b,a]])/Delta0
assert all(sp.cancel(x)==0 for x in (S0*S0inv-sp.eye(2)))

print("Scalar sector inverse: PASS")
print("Delta0 registered symbolically: PASS")


Scalar sector inverse: PASS
Delta0 registered symbolically: PASS


## 4. Reconstruction analytique de \(S_6^{-1}\)

Sur la branche
\[
\Delta_0\lambda_\Delta\lambda_V\neq0,
\]
\[
S_{\rm sec}^{-1}
=
S_0^{-1}\oplus
\lambda_\Delta^{-1}\oplus
\lambda_{12}^{-1}\oplus
\lambda_V^{-1}\oplus
\lambda_V^{-1}.
\]

Puis
\[
\boxed{S_6^{-1}=M S_{\rm sec}^{-1}M^T}.
\]


In [5]:
Ssec_inv=sp.zeros(6,6)
for i in range(2):
    for j in range(2):
        Ssec_inv[i,j]=S0inv[i,j]
Ssec_inv[2,2]=1/lamD
Ssec_inv[3,3]=1/lam12
Ssec_inv[4,4]=1/lamV
Ssec_inv[5,5]=1/lamV

S6inv=M*Ssec_inv*M.T

# Exact generic identity for the six-dimensional sector.
res6=S6*S6inv-sp.eye(6)
assert all(sp.cancel(x)==0 for x in res6)

print("Generic S6 inverse: PASS")
print("S6*S6^{-1}=I6: PASS")


Generic S6 inverse: PASS
S6*S6^{-1}=I6: PASS


## 5. Reconstruction exacte de \(Q_{\rm total}^{-1}\)

Avec
\[
Q=
\begin{pmatrix}
A_6&B\\B^T&D_4
\end{pmatrix},
\qquad
S_6=A_6-BD_4^{-1}B^T,
\]
la formule exacte est
\[
\boxed{
Q^{-1}
=
\begin{pmatrix}
S_6^{-1}&-S_6^{-1}BD_4^{-1}\\
-D_4^{-1}B^TS_6^{-1}&
D_4^{-1}+D_4^{-1}B^TS_6^{-1}BD_4^{-1}
\end{pmatrix}.
}
\]

Tous les blocs du membre de droite sont désormais explicitement connus. L'expansion composante par composante d'une matrice rationnelle \(10\times10\) n'est pas nécessaire pour définir l'inverse.


In [6]:
# Keep the exact block representation rather than forcing a huge expansion.
Qinv_blocks={
    "UL":"S6inv",
    "UR":"-S6inv*B64*D4inv",
    "LL":"-D4inv*B64.T*S6inv",
    "LR":"D4inv + D4inv*B64.T*S6inv*B64*D4inv"
}
print("Generic Q_total inverse block formula: REGISTERED")


Generic Q_total inverse block formula: REGISTERED


## 6. Contrôle indépendant sur deux témoins rationnels

La formule bloc est testée sur deux points distincts de la branche non dégénérée.


In [7]:
witness_list=[
 {c1:sp.Rational(2,5),c2:sp.Rational(1,7),c3:sp.Rational(-1,11),c4:sp.Rational(3,13),
  s:sp.Rational(5,4),r:sp.Rational(2,7)},
 {c1:sp.Rational(1,3),c2:sp.Rational(-1,8),c3:sp.Rational(1,10),c4:sp.Rational(2,9),
  s:sp.Rational(6,5),r:sp.Rational(3,8)}
]

for n,w in enumerate(witness_list,1):
    Qw=sp.Matrix(Q.subs(w))
    A6w=Qw[:6,:6]; Bw=Qw[:6,6:10]; Dw=Qw[6:10,6:10]
    Diw=Dw.inv()
    Sw=A6w-Bw*Diw*Bw.T
    Siw=Sw.inv()
    Qiw=sp.Matrix.vstack(
        sp.Matrix.hstack(Siw,-Siw*Bw*Diw),
        sp.Matrix.hstack(-Diw*Bw.T*Siw,Diw+Diw*Bw.T*Siw*Bw*Diw)
    )
    assert Qw.rank()==10
    assert Qw*Qiw==sp.eye(10)
    print(f"Witness {n}: rank10 + exact inverse identity PASS")


Witness 1: rank10 + exact inverse identity PASS
Witness 2: rank10 + exact inverse identity PASS


## 7. Relèvement vers \(v_i\) arbitraire

Pour tout \(v_i\neq0\), il existe localement \(R\in SO(3)\) tel que
\[
Rv=(0,0,\sqrt{v^2}).
\]

La représentation induite \(T(R)\) agit sur \((K_{ij},S,W_i)\), et
\[
\boxed{
Q^{-1}(v)
=
T(R)^{-1}
Q_{\rm aligned}^{-1}(\sqrt{v^2})
T(R)^{-T}.
}
\]

La dégénérescence double du secteur vectoriel et la structure du secteur transverse assurent l'indépendance vis-à-vis de la rotation résiduelle autour de \(v_i\).

La branche \(v_i=0\) est obtenue directement en posant \(r=0\).


In [8]:
# v=0 control on a generic rational point
w0={c1:sp.Rational(2,5),c2:sp.Rational(1,7),c3:sp.Rational(-1,11),
    c4:sp.Rational(3,13),s:sp.Rational(5,4),r:0}
Q0=sp.Matrix(Q.subs(w0))
assert Q0.rank()==10
assert Q0*Q0.inv()==sp.eye(10)
print("v_i=0 branch control: PASS")


v_i=0 branch control: PASS


## 8. Densité normale complète et fermeture de \(R_{DD2}\)

Avec \(J\) et \(U\) déjà reconstruits par 7.7.2.4 :
\[
\boxed{
\mathcal C_\perp
=
\frac12(P-J)^TQ_{\rm total}^{-1}(P-J)-U.
}
\]

La contrainte de shift est déjà explicite :
\[
\boxed{
\mathcal C_i=
-2h_{ij}D_k\pi^{kj}
+p_sD_is
+p_v^{\,j}D_iv_j
-D_j(p_v^{\,j}v_i).
}
\]

Le verrou \(R_{DD2}\) est donc fermé **sur la branche générique non dégénérée**.


In [9]:
GATES={
 "SO2_Schur_sector_decomposition":True,
 "scalar_2x2_inverse_exact":True,
 "generic_S6_inverse_explicit":True,
 "S6_inverse_identity_symbolic":True,
 "generic_Qtotal_inverse_block_formula":True,
 "two_independent_rank10_inverse_witnesses":True,
 "v_zero_branch_control":True,
 "full_Cperp_generic_constructive_form":True,
 "full_Ci_explicit_generic":True,
 "RDD2_computed":True,
 "RDD2_generic_branch_closed":True,
 "all_degeneracy_surfaces_globally_resolved":False,
 "hypersurface_algebra_closed":False,
}

for k,v in GATES.items():
    print(k,":",v)

FINAL_STATUS=(
 "PASS-ANALYTIC-SCHUR-SECTOR-DECOMPOSITION-AND-GENERIC-S6-INVERSE_"
 "GENERIC-QTOTAL-INVERSE-CONSTRUCTED-BY-EXACT-BLOCK-FORMULA_"
 "RDD2-CLOSED-ON-GENERIC-NONDEGENERATE-BRANCH_"
 "HYPERSURFACE-ALGEBRA-STILL-OPEN"
)
DISPERSION_READY=False

assert GATES["RDD2_computed"]
assert not GATES["hypersurface_algebra_closed"]
assert DISPERSION_READY is False

print("\nFINAL STATUS:",FINAL_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


SO2_Schur_sector_decomposition : True
scalar_2x2_inverse_exact : True
generic_S6_inverse_explicit : True
S6_inverse_identity_symbolic : True
generic_Qtotal_inverse_block_formula : True
two_independent_rank10_inverse_witnesses : True
v_zero_branch_control : True
full_Cperp_generic_constructive_form : True
full_Ci_explicit_generic : True
RDD2_computed : True
RDD2_generic_branch_closed : True
all_degeneracy_surfaces_globally_resolved : False
hypersurface_algebra_closed : False

FINAL STATUS: PASS-ANALYTIC-SCHUR-SECTOR-DECOMPOSITION-AND-GENERIC-S6-INVERSE_GENERIC-QTOTAL-INVERSE-CONSTRUCTED-BY-EXACT-BLOCK-FORMULA_RDD2-CLOSED-ON-GENERIC-NONDEGENERATE-BRANCH_HYPERSURFACE-ALGEBRA-STILL-OPEN
DISPERSION_READY = False


## 9. Étape suivante autorisée

La chaîne 7.7.2 a maintenant atteint son critère de sortie sur la branche générique.

### `0.3.2.7.3.7.3 — Full Hypersurface Constraint Algebra Closure`

Elle devra calculer
\[
\{\mathcal C_i,\mathcal C_j\},
\qquad
\{\mathcal C_\perp,\mathcal C_i\},
\qquad
\{\mathcal C_\perp,\mathcal C_\perp\},
\]
et décider si
\[
\boxed{R_{DD3}=0}.
\]

La dispersion reste différée.


In [10]:
artifact={
 "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.6",
 "final_status":FINAL_STATUS,
 "generic_S6_inverse_explicit":True,
 "generic_Qtotal_inverse":"exact Schur block formula with explicit S6inv and D4inv",
 "RDD2_status":"CLOSED_GENERIC_NONDEGENERATE_BRANCH",
 "gates":GATES,
 "dispersion_ready":False,
 "next":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3_Full_Hypersurface_Constraint_Algebra_Closure.ipynb"
}
export_dir=Path("/content/gvh_exports") if Path("/content").exists() else Path.cwd()/"gvh_exports"
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path=export_dir/"gvh_0.3.2.7.3.7.2.6_schur_sector_inverse.json"
artifact_path.write_text(json.dumps(artifact,indent=2),encoding="utf-8")
print("Artifact:",artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.2.6_schur_sector_inverse.json


# Conclusion

La décomposition sectorielle réduit le problème \(6\times6\) à :
\[
\boxed{
2\times2\ \oplus\ 1\ \oplus\ 1\ \oplus\ 1\ \oplus\ 1.
}
\]

L'inverse \(S_6^{-1}\) est obtenue analytiquement et vérifiée symboliquement :
\[
\boxed{S_6S_6^{-1}=I_6}.
\]

L'inverse totale est ensuite reconstruite exactement par la formule de Schur.

Ainsi :
\[
\boxed{\texttt{RDD2\_computed=True}}
\]
et
\[
\boxed{R_{DD2}\text{ CLOSED — generic nondegenerate branch}}.
\]

Le prochain verrou est \(R_{DD3}\), pas la dispersion :
\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]
